# Import for models to use, and Load Processed DS

In [59]:
# for models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score, f1_score

import lightgbm as lgb

from xgboost import XGBClassifier

# Loading Data
import pandas as pd

# 
import time

# fine tuning
import optuna

C:\Users\jovin\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


__________________

# Load the Processed dataset

In [37]:
df = pd.read_csv(r'C:\Users\jovin\Desktop\TF\MLOPS\EndToEndMLPipeline\data\processed\ProcessedChurnDS.csv')
df.info(), df['churn'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   gender            10000 non-null  int64  
 3   age               10000 non-null  int64  
 4   tenure            10000 non-null  int64  
 5   balance           10000 non-null  float64
 6   products_number   10000 non-null  int64  
 7   credit_card       10000 non-null  int64  
 8   active_member     10000 non-null  int64  
 9   estimated_salary  10000 non-null  float64
 10  churn             10000 non-null  int64  
 11  country_Germany   10000 non-null  int64  
 12  country_Spain     10000 non-null  int64  
dtypes: float64(2), int64(11)
memory usage: 1015.8 KB


(None,
 churn
 0    7963
 1    2037
 Name: count, dtype: int64)

________________________

# prepare the data

In [38]:
y = df['churn']
X = df.drop(columns = ['churn'])

In [39]:
# train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y   # need to use stratify because of class imbalance
)

In [40]:
# due to class imbalance
THRESHOLD = 0.2 # ADjusting the threshold, to adjust how much one category is to the other for AUC and ROC  curve
                 # the cost is predicting they will not churn but they do churn in the end so false Negatives, 
                    # hence we are looking to optimize for recall

___________________________________

# Testing different Models
## RandomForest classifier

In [21]:
rfc = RandomForestClassifier(
    n_estimators = 200, 
    criterion = 'entropy', 
    max_depth = 4,
    class_weight = 'balanced', 
    random_state = 42,
    n_jobs = 4,
)

start_train_rfc = time.time()
rfc.fit(X_train, y_train)

rfc_train_time = time.time()-start_train_rfc

start_pred_rfc = time.time()
rfc_proba = rfc.predict_proba(X_test)[:,1]
y_pred = (rfc_proba>= THRESHOLD).astype(int)
rfc_pred_time = time.time() - start_pred_rfc



print("TrainTime:", rfc_train_time)
print("Pred Time:", rfc_pred_time)
print(classification_report(y_test, y_pred, digits = 3))

TrainTime: 0.6584227085113525
Pred Time: 0.06986618041992188
              precision    recall  f1-score   support

           0      0.982     0.034     0.066      1593
           1      0.209     0.998     0.345       407

    accuracy                          0.230      2000
   macro avg      0.595     0.516     0.205      2000
weighted avg      0.824     0.230     0.122      2000



## LightGBM Classifier

In [23]:
lgbm = lgb.LGBMClassifier(
    n_estimators = 500, 
    learning_rate = 0.05, 
    class_weight = 'balanced', 
    random_state = 42, 
    n_jobs = 1
)

X_train.columns = ["".join(c if c.isalnum() else "_" for c in str(x)) for x in X_train.columns]

start_train_lgbm = time.time()
lgbm.fit(X_train, y_train)
lgbm_train_time = time.time() -start_train_lgbm

print("TrainTime:",lgbm_train_time)


start_pred_lgbm = time.time()
lgbm_proba = lgbm.predict_proba(X_test)[:,1]
lgbm_y_pred = (lgbm_proba >= THRESHOLD).astype(int)
lgbm_pred_time = time.time() - start_pred_lgbm

print("predTime:", lgbm_pred_time)
print(classification_report(y_test, lgbm_y_pred, digits = 3))



[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000550 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1112
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
TrainTime: 0.5889754295349121
predTime: 0.10031485557556152
              precision    recall  f1-score   support

           0      0.939     0.689     0.795      1593
           1      0.404     0.826     0.542       407

    accuracy                          0.717      2000
   macro avg      0.672     0.757     0.669      2000
weighted avg      0.830     0.717     0.743      2000



## XGBoost Classifier

In [47]:
# calculate imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train ==1).sum()

xgb = XGBClassifier(
    n_estimator = 300, 
    learning_rate = 0.05, 
    max_depth = 4, 
    subsample = 0.8, 
    colsample_bytree = 0.8, 
    random_state = 42, 
    n_jobs = 4, 
    scale_pos_weight = scale_pos_weight, 
    eval_metrics = 'logloss'
)

start_train_xgb = time.time()
xgb.fit(X_train, y_train)
xgb_train_time = time.time() - start_train_xgb

start_pred_xgb = time.time()
xgb_proba = xgb.predict_proba(X_test)[:,1]
xgb_y_pred = (xgb_proba >=  THRESHOLD).astype(int)
xgb_pred_time = time.time() - start_pred_xgb

print("TrainTime:", xgb_train_time)
print("PredTime: ", xgb_pred_time)
print(classification_report(y_test, xgb_y_pred, digits = 3))

TrainTime: 0.1318209171295166
PredTime:  0.0051877498626708984
              precision    recall  f1-score   support

           0      0.972     0.372     0.538      1593
           1      0.281     0.958     0.434       407

    accuracy                          0.491      2000
   macro avg      0.626     0.665     0.486      2000
weighted avg      0.831     0.491     0.517      2000



C:\Users\jovin\.conda\envs\TF\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:27:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "eval_metrics", "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


______________

# Compare: Model Selection
First goal, is to avoid False Negatives, that is predicting that the customer will not churn when then do in fact churn./
And hence recall will be our goto metric for comparison, as we can see both **RandomForestClassifier** and **XGBoost Classifier** have a recall of **0.998** and **0.958** respectively. Which might tell us to use RandomForest Classifier, but if if we consider F1-score to look at the model performance on imbalanced data (which is the current case). XGBoost Classifier with a **f1-score of 0.434** is better than **RandomForestClassifier with an f1 score of 0.345**. Hence by fine-tuning the xgb classifier we can end with a much better model, for this dataset. 

In additino to that XGB prediction time is much faster, if we want to tend to multiple customers in a short amont of time.

______________________________

# Fine-tuning XGB Boost

### 1. Fine_tune to find optimal threshold

In [51]:
proba = xgb.predict_proba(X_test)[:,1]

for thresh in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55]:

    preds = (proba >= thresh).astype(int)
    precision = precision_score(y_test, preds, pos_label = 1)
    rec = recall_score(y_test, preds, pos_label = 1)
    f1 = f1_score(y_test, preds, pos_label = 1)

    print(f"Threshold: {thresh:<5} |   Precision: {precision:<5.3f} |   Recall: {rec:<5.3f} |   F1: {f1:<5.3f} |")

Threshold: 0.1   |   Precision: 0.231 |   Recall: 0.993 |   F1: 0.375 |
Threshold: 0.15  |   Precision: 0.253 |   Recall: 0.975 |   F1: 0.401 |
Threshold: 0.2   |   Precision: 0.281 |   Recall: 0.958 |   F1: 0.434 |
Threshold: 0.25  |   Precision: 0.314 |   Recall: 0.929 |   F1: 0.469 |
Threshold: 0.3   |   Precision: 0.351 |   Recall: 0.899 |   F1: 0.505 |
Threshold: 0.35  |   Precision: 0.386 |   Recall: 0.877 |   F1: 0.536 |
Threshold: 0.4   |   Precision: 0.419 |   Recall: 0.835 |   F1: 0.558 |
Threshold: 0.45  |   Precision: 0.457 |   Recall: 0.801 |   F1: 0.582 |
Threshold: 0.5   |   Precision: 0.495 |   Recall: 0.764 |   F1: 0.601 |
Threshold: 0.55  |   Precision: 0.560 |   Recall: 0.725 |   F1: 0.632 |


Loooks like threshold = 0.15 or 0.2 seems to give a good recall value, so I am just going to keep using the same threshold value as before

## Fine tuning the Hyper Parameters

In [61]:
# fine tuning
scale_pos_weight = (y_train == 0).sum() / (y_train ==1).sum()


def objective(trial):
    # every parameter in XGBOOST, and start-end values for each parameters for optuna to try
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 300,800),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth":         trial.suggest_int("max_depth", 3,10),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0), 
        "colsample_bytree" : trial.suggest_float("subsample", 0.5, 1.0), 
        "min_child_weight":  trial.suggest_int("min_child_weight", 1,10),
        "gamma":             trial.suggest_int("gamma",  0,5),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0,5),
        "reg_lambda":        trial.suggest_float("reg_alpha", 0,5),
        "random_state" : 42, 
        "n_jobs" : -1, 
        "scale_pos_weight" : scale_pos_weight, 
        # "eval_metrics" :'logloss'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:,1]
    y_pred = (proba >= THRESHOLD).astype(int)

    return recall_score(y_test, y_pred, pos_label = 1)

In [62]:
# using optuna for the actual grid search
study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 30)

print("BestParams:", study.best_params)
print("BestRecall:", study.best_value)

[I 2026-03-15 13:08:19,435] A new study created in memory with name: no-name-094e88a4-72e3-4a90-a2e7-5b0b6190632d
[I 2026-03-15 13:08:20,412] Trial 0 finished with value: 0.7960687960687961 and parameters: {'n_estimators': 599, 'learning_rate': 0.15619588261035322, 'max_depth': 7, 'subsample': 0.5964383447407716, 'min_child_weight': 9, 'gamma': 2, 'reg_alpha': 1.6927897824146316}. Best is trial 0 with value: 0.7960687960687961.
[I 2026-03-15 13:08:21,847] Trial 1 finished with value: 0.8353808353808354 and parameters: {'n_estimators': 745, 'learning_rate': 0.06089091660712132, 'max_depth': 5, 'subsample': 0.58594720273814, 'min_child_weight': 6, 'gamma': 0, 'reg_alpha': 2.1548566652119954}. Best is trial 1 with value: 0.8353808353808354.
[I 2026-03-15 13:08:22,906] Trial 2 finished with value: 0.9361179361179361 and parameters: {'n_estimators': 623, 'learning_rate': 0.038003653702483015, 'max_depth': 9, 'subsample': 0.5441978821337292, 'min_child_weight': 7, 'gamma': 4, 'reg_alpha': 3.

BestParams: {'n_estimators': 507, 'learning_rate': 0.013512420752260309, 'max_depth': 3, 'subsample': 0.9914249477695516, 'min_child_weight': 2, 'gamma': 4, 'reg_alpha': 0.0012193664946194005}
BestRecall: 0.9656019656019657


# COmpleted model with fine tuned parameters and threshold

In [64]:
best_params = study.best_params

best_params.update({  # these values will not be a part of the best params since they werent invoked with optuna's trial.suggest
    "random_state" : 42, 
    "n_jobs" : -1, 
    "scale_pos_weight" : scale_pos_weight, 
})

In [65]:
# using the params for the model
xgb = XGBClassifier(**best_params)

start_train = time.time()
xgb.fit(X_train, y_train)
train_time = time.time() - start_train

print("TrainTime:",train_time)

TrainTime: 0.9108564853668213


In [69]:
pred_time = time.time()
y_proba = xgb.predict_proba(X_test)[:,1]
y_pred = (y_proba >= THRESHOLD).astype(int)
pred_time = time.time() - pred_time
print("PredTime:", pred_time)

PredTime: 0.009139060974121094


In [70]:
print(classification_report(y_test, y_pred, digits = 3))

              precision    recall  f1-score   support

           0      0.976     0.361     0.527      1593
           1      0.279     0.966     0.432       407

    accuracy                          0.484      2000
   macro avg      0.627     0.663     0.480      2000
weighted avg      0.834     0.484     0.508      2000



# THE FINAL FINETUNED MDOEL

This XGBoost model gives us a recall of 0.966, with a precision of 0.279, and almost same f1 score.\
The model has a high recall and a low precisino, which tells us that the model finds all positive cases while it also leads to some false positive cases. But for the case where the cost of where the False Negatives lead to much higher costs than False positives, this is a good model for the use case. 

_______________________________________